[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adaptionlabs/adaption-runbooks/blob/main/autoscientist/alignment_training.ipynb)

# Train with Alignment (SFT → DPO)

Build a preference-pair dataset, then run AutoScientist's alignment pipeline (supervised fine-tuning followed by DPO) and download the aligned checkpoint.

**Docs:** [AutoScientist overview](https://docs.adaptionlabs.ai/autoscientist/overview/) · [Train on preference pairs](https://docs.adaptionlabs.ai/autoscientist/running-autoscientist/#train-on-preference-pairs)  
**Prerequisites:** an Adaption API key ([create one](https://adaptionlabs.ai/app/settings?tab=api_keys)) set as `ADAPTION_API_KEY` (env var or Colab secret).

In [ ]:
%pip install -q adaption

## Setup

Load your API key. In Colab, add `ADAPTION_API_KEY` to the secrets panel (🔑 in the sidebar) and enable it for this notebook. Running locally, the cell prompts for the key if the environment variable is not set.

In [ ]:
import os

from adaption import Adaption

if "ADAPTION_API_KEY" not in os.environ:
    try:
        from google.colab import userdata  # Colab secrets

        os.environ["ADAPTION_API_KEY"] = userdata.get("ADAPTION_API_KEY")
    except ImportError:
        from getpass import getpass

        os.environ["ADAPTION_API_KEY"] = getpass("ADAPTION_API_KEY: ")

client = Adaption()

## Parameters

In [ ]:
DATASET_URL = "https://huggingface.co/datasets/tatsu-lab/alpaca"
DATASET_FILES = ["data/train-00000-of-00001-a09b74b3ef9c3b56.parquet"]
COLUMN_MAPPING = {
    "prompt": "instruction",
    "completion": "output",
    "context": ["input"],
}

# Alpaca has ~52k rows. Adapt only a slice so the demo builds quickly; set to
# None to use the whole dataset.
MAX_ROWS = 1_000

# DPO requires at least 12,000 effective preference pairs, including augmentation.
TARGET_PREFERENCE_PAIRS = 12_000

## Step 1 — Import a dataset

In [ ]:
dataset = client.datasets.create(
    source={"url": DATASET_URL, "files": DATASET_FILES},
)
dataset_id = dataset.dataset_id
print("dataset_id:", dataset_id)

import time

while True:
    status = client.datasets.get_status(dataset_id)
    if status.status == "failed":
        err = status.error_data
        msg = (err and err.message) or "unknown error"
        raise RuntimeError(f"Ingestion failed: {msg}")
    if status.row_count is not None:
        print(f"Imported {status.row_count} rows")
        break
    time.sleep(5)

## Step 2 — Build a preference-pair dataset

Alignment (DPO) needs a dataset prepared as **preference pairs**: a chosen and a rejected response per prompt. Run Adaptive Data with `training_type="preference_pairs"` to generate them from the instruction data. We cap the run at `MAX_ROWS` so the demo builds quickly.

In [ ]:
job_specification = {"max_rows": MAX_ROWS} if MAX_ROWS else {}

estimate = client.datasets.adapt(
    dataset_id,
    column_mapping=COLUMN_MAPPING,
    training_type="preference_pairs",
    job_specification=job_specification,
    estimate=True,
)
print(f"Estimated credits: {estimate.estimated_credits_consumed}")

client.datasets.adapt(
    dataset_id,
    column_mapping=COLUMN_MAPPING,
    training_type="preference_pairs",
    job_specification=job_specification,
)
status = client.datasets.wait_for_completion(dataset_id, timeout=3600)
if status.status == "failed":
    raise RuntimeError(f"Adaptive Data failed: {status.error_data}")
print("Preference-pair dataset ready:", status.status)

## Step 3 — Plan augmentation (domain + general rows)

Augmentation grows the preference set before training — its real purpose is **coverage**, not just hitting a row count. Two kinds of rows matter here (see the [Expand a Dataset](../adaptive-data/expand_dataset.ipynb) runbook for a closer look):

- **Domain rows** add more tasks from your dataset's own domain, so fewer tasks are unfamiliar to the model and it generalizes better on the behavior you want.
- **General rows** add tasks from other domains to preserve the model's broad capability and keep the training mix balanced (a common target is ~40% general).

Requesting enough of both also clears DPO's 12,000-pair minimum.

In [ ]:
dataset = client.datasets.get(dataset_id=dataset_id)
have = dataset.row_count or 0

# Domain rows reinforce your dataset's own domain; general rows add other domains
# for balance. Sized here to clear the 12,000-pair minimum with ~40% general rows.
augmentation_domain_rows = max(0, TARGET_PREFERENCE_PAIRS - have)
base = have + augmentation_domain_rows
augmentation_general_rows = round(base * 0.4 / 0.6)
print(
    f"Have {have} adapted pairs; adding {augmentation_domain_rows} domain "
    f"+ {augmentation_general_rows} general pairs via augmentation"
)

## Step 4 — Start the alignment run

Pass `training_method="alignment"` **explicitly** — a preference-pair dataset does not select alignment on its own. If the dataset was not prepared as preference pairs, the API resolves the run to `instruction`, so read `run.training_method` to confirm.

In [ ]:
run = client.autoscientist.create(
    dataset_id=dataset_id,
    training_method="alignment",
    augmentation_domain_rows=augmentation_domain_rows,
    augmentation_general_rows=augmentation_general_rows,
)
print(run.id, run.status, "| resolved:", run.training_method)

if run.training_method != "alignment":
    raise RuntimeError(
        f"Run resolved to {run.training_method!r}; the dataset was not "
        "prepared as preference_pairs."
    )

## Step 5 — Wait for completion (SFT → DPO)

Alignment is one run that trains an SFT adapter and then a DPO stage under the same ID. During the handoff the run stays `running`, `best_win_rate` can be null, and `download_available` is false — keep polling. A timeout stops waiting, not training; call `wait_for_completion` again to resume tracking.

In [ ]:
from adaption import TrainingTimeout

try:
    run = client.autoscientist.wait_for_completion(run.id)
except TrainingTimeout as exc:
    print("Still running:", exc.resource_id, exc.last_status)
    raise

if run.status != "succeeded":
    raise RuntimeError(f"Alignment run ended with status: {run.status}")

print("Iterations:", run.iterations_completed, "/", run.max_iterations)
print("Best win rate:", run.best_win_rate)

## Step 6 — Download the aligned checkpoint

The download comes from the DPO stage's best iteration and streams a compressed tar archive.

In [ ]:
if not run.download_available:
    raise RuntimeError("The checkpoint is not available for download")

with client.autoscientist.with_streaming_response.download(run.id) as response:
    response.stream_to_file("aligned-checkpoint.tgz")
print("Wrote aligned-checkpoint.tgz")

## Next steps

- [Interpret the results](https://docs.adaptionlabs.ai/autoscientist/interpreting-results/) — win rate, loss, and the DPO stage transition.
- [Tune hyperparameters](https://docs.adaptionlabs.ai/autoscientist/recommended-hyperparameters/) such as `dpo_beta`.

## Clean up

To stop a run that is still in flight, cancel it by ID:

In [ ]:
# client.autoscientist.cancel(run.id)